## 0. 환경설정

In [54]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python pandas gradio python-dotenv -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 환경변수 로드

import pandas as pd
import mysql.connector
import gradio as gr

from neo4j import GraphDatabase, basic_auth
from neo4j.time import Date

from neo4j_graphrag.retrievers import Text2CypherRetriever
try:
    from neo4j_graphrag.llm import OpenAILLM
except ImportError:
    from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG
import re
import time

import requests
from math import radians, cos, sin, asin, sqrt, atan2

print("모듈 로드 완료")

모듈 로드 완료


In [56]:
# OPENAI_API_KEY는 .env에서 자동 로드됨
# 확인용:
print("OpenAI API Key 설정됨:", bool(os.environ.get("OPENAI_API_KEY")))

OpenAI API Key 설정됨: True


In [57]:
# KAKAO_REST_API_KEY는 .env에서 자동 로드됨
# 확인용:
print("Kakao API Key 설정됨:", bool(os.environ.get("KAKAO_REST_API_KEY")))

Kakao API Key 설정됨: True


In [58]:
api_key  = os.environ.get("KAKAO_REST_API_KEY")
print("exists:", bool(api_key))
print("repr:", repr(api_key))
print("prefix:", api_key[:8] if api_key else None)

exists: True
repr: 'aa921e29bd619d0b1ad79ece4078f34b'
prefix: aa921e29


## 0-1. MY SQL 및 NEO4J 불러오기

In [59]:
MYSQL_HOST = os.environ.get("MYSQL_HOST", "localhost")
MYSQL_USER = os.environ.get("MYSQL_USER", "")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")
MYSQL_DB = os.environ.get("MYSQL_DB", "")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT", 3306))

In [60]:
NEO4J_URI = os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PW", "")

In [61]:
COUPLE_ID = 2 # 커플 아이디 임시

In [62]:
conn = None
if all([MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB]):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            database=MYSQL_DB,
            port=MYSQL_PORT
        )
        print("MySQL 연결 성공")
    except Exception as e:
        conn = None
        print(f"MySQL 연결 실패: {e}")
else:
    print("MySQL 미설정 (.env 확인 필요)")

MySQL 연결 성공


### 0-1-1. 사용자 선호 조회

In [63]:
query = """
SELECT *
FROM COUPLE_PREFERENCES
WHERE couple_id = %s
"""

df_pref = pd.read_sql(query, conn, params=(COUPLE_ID,))

if df_pref.empty:
    print(f"couple_id={COUPLE_ID} 선호 데이터 없음 (더미 사용)")
    pref = {"couple_id": COUPLE_ID, "region": "서울", "budget": "300만원"}
else:
    pref = df_pref.iloc[0].to_dict()
    print("=== 사용자 선호 ===")
    print(pref)
    print("=== 컬럼 목록 ===")
    print(df_pref.columns.tolist())


couple_id=2 선호 데이터 없음 (더미 사용)


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_26648\3896295995.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pref = pd.read_sql(query, conn, params=(COUPLE_ID,))


### 0-1-2. 사용자 취향

In [64]:
def get_user_preference(conn, couple_id):
    query = """
    SELECT *
    FROM COUPLE_PREFERENCES
    WHERE couple_id = %s
    """

    df = pd.read_sql(query, conn, params=(couple_id,))

    if df.empty:
        return None
    
    if df_pref.empty:
        raise ValueError(f"COUPLE_PREFERENCES에 couple_id={COUPLE_ID} 데이터가 없습니다.")

    return df.iloc[0].to_dict()

### 0-1-3. 좋아요

In [65]:
def get_user_likes(conn, couple_id):
    query = """
    SELECT *
    FROM COUPLE_VENUE_LIKES
    WHERE couple_id = %s
    """
    df = pd.read_sql(query, conn, params=(couple_id,))

    if df.empty:
        return []

    return df.to_dict(orient="records")

### 0-1-4. 좋아요 누른 홀이름

In [66]:
def get_liked_halls_with_names(conn, couple_id):
    query = """
    SELECT l.*, h.name
    FROM COUPLE_VENUE_LIKES l
    JOIN WEDDING_HALL h ON l.partner_id = h.partnerId
    WHERE l.couple_id = %s
    """
    df = pd.read_sql(query, conn, params=(couple_id,))

    if df.empty:
        return []

    return df.to_dict(orient="records")

### 0-1-5. 좋아요 기반 추천(+ 평점 순)

In [67]:
def recommend_by_likes(conn, couple_id):
    query = """
    SELECT
        h2.name,
        h2.rating,
        COUNT(*) AS score
    FROM couple_venue_likes l1
    JOIN couple_venue_likes l2
        ON l1.couple_id = l2.couple_id
    JOIN wedding_hall h2
        ON l2.hall_id = h2.hall_id
    WHERE l1.hall_id IN (
        SELECT hall_id
        FROM couple_venue_likes
        WHERE couple_id = %s
    )
    AND l2.hall_id NOT IN (
        SELECT hall_id
        FROM couple_venue_likes
        WHERE couple_id = %s
    )
    AND h2.rating >= 4.3
    GROUP BY h2.hall_id, h2.name, h2.rating
    ORDER BY score DESC
    LIMIT 5
    """

    df = pd.read_sql(query, conn, params=(couple_id, couple_id))
    return df.to_dict(orient="records")

In [68]:
def recommend_by_likes_with_region_boost(conn, couple_id, preferred_region=None, preferred_sub_region=None):
    query = """
    SELECT
        h2.hall_id,
        h2.name,
        h2.region,
        h2.sub_region,
        h2.rating,
        COUNT(*) AS like_score,
        CASE
            WHEN %s IS NOT NULL AND h2.region = %s THEN 1
            ELSE 0
        END
        +
        CASE
            WHEN %s IS NOT NULL AND h2.sub_region = %s THEN 2
            ELSE 0
        END AS region_score
    FROM couple_venue_likes l1
    JOIN couple_venue_likes l2
        ON l1.couple_id = l2.couple_id
    JOIN wedding_hall h2
        ON l2.hall_id = h2.hall_id
    WHERE l1.hall_id IN (
        SELECT hall_id
        FROM couple_venue_likes
        WHERE couple_id = %s
    )
    AND l2.hall_id NOT IN (
        SELECT hall_id
        FROM couple_venue_likes
        WHERE couple_id = %s
    )
    AND h2.rating >= 4.3
    GROUP BY h2.hall_id, h2.name, h2.region, h2.sub_region, h2.rating
    ORDER BY (like_score + region_score) DESC, h2.rating DESC
    LIMIT 5
    """

    params = (
        preferred_region, preferred_region,
        preferred_sub_region, preferred_sub_region,
        couple_id, couple_id
    )

    df = pd.read_sql(query, conn, params=params)
    return df.to_dict(orient="records")

## NEO4J 연결

In [69]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD)
)

In [70]:
with driver.session() as session:
    count_result = session.run("MATCH (n) RETURN count(n) AS cnt")
    print("Neo4j node count:", count_result.single()["cnt"])

Neo4j node count: 4667


## 스키마 함수 추출

In [71]:
def get_node_datatype(value):
    if isinstance(value, str):
        return "STRING"
    elif isinstance(value, int):
        return "INTEGER"
    elif isinstance(value, float):
        return "FLOAT"
    elif isinstance(value, bool):
        return "BOOLEAN"
    elif isinstance(value, list):
        return f"LIST[{get_node_datatype(value[0])}]" if value else "LIST"
    elif isinstance(value, Date):
        return "DATE"
    else:
        return "UNKNOWN"

In [72]:
def get_schema(uri, user, password):
    driver_local = GraphDatabase.driver(uri, auth=basic_auth(user, password))

    with driver_local.session() as session:
        node_query = """
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        """

        rel_query = """
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        """

        rel_direction_query = """
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS start_label, type(r) AS rel_type, labels(b) AS end_label
        ORDER BY start_label, rel_type, end_label
        """

        nodes = session.run(node_query)
        relationships = session.run(rel_query)
        rel_directions = session.run(rel_direction_query)

        schema = {
            "nodes": {},
            "relationships": {},
            "relations": []
        }

        for record in nodes:
            label = record["label"]
            key = record["key"]
            sample_value = record["sample_value"]
            inferred_type = get_node_datatype(sample_value)

            if label not in schema["nodes"]:
                schema["nodes"][label] = {}
            schema["nodes"][label][key] = inferred_type

        for record in relationships:
            rel_type = record["rel_type"]
            key = record["key"]
            sample_value = record["sample_value"]
            inferred_type = get_node_datatype(sample_value)

            if rel_type not in schema["relationships"]:
                schema["relationships"][rel_type] = {}
            schema["relationships"][rel_type][key] = inferred_type

        for record in rel_directions:
            start_labels = record["start_label"]
            end_labels = record["end_label"]

            if not start_labels or not end_labels:
                continue

            start_label = start_labels[0]
            rel_type = record["rel_type"]
            end_label = end_labels[0]

            schema["relations"].append(f"(:{start_label})-[:{rel_type}]->(:{end_label})")

    driver_local.close()
    return schema

In [73]:
def format_schema(schema):
    result = []

    result.append("Node properties:")
    for label, properties in schema["nodes"].items():
        props = ", ".join(f"{k}: {v}" for k, v in properties.items())
        result.append(f"{label} {{{props}}}")

    result.append("Relationship properties:")
    for rel_type, properties in schema["relationships"].items():
        props = ", ".join(f"{k}: {v}" for k, v in properties.items())
        result.append(f"{rel_type} {{{props}}}")

    result.append("The relationships:")
    for relation in schema["relations"]:
        result.append(relation)

    return "\n".join(result)


schema = get_schema(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
neo4j_schema = format_schema(schema)

print("=== Neo4j Schema ===")
print(neo4j_schema)

=== Neo4j Schema ===
Node properties:
Image {url: STRING, title: STRING}
Hall {partnerId: INTEGER, name: STRING, address: STRING, profileUrl: STRING, rating: FLOAT, region: STRING, reviewCnt: INTEGER, tel: STRING, typeName: STRING, uuid: STRING, minRentalPrice: INTEGER, subRegion: STRING, partnerProfileId: INTEGER, modTsp: STRING, minMealPrice: INTEGER, partnerProfileName: STRING, minIndividualHallPrice: INTEGER, maxRentalPrice: INTEGER, bookingState: INTEGER, maxMealPrice: INTEGER, memoContent: STRING, partnerProfileUuid: STRING, availableContract: INTEGER, maxIndividualHallPrice: INTEGER}
EventTag {code: STRING}
District {name: STRING}
Tag {name: STRING, typeName: STRING, type: INTEGER, category: STRING}
Region {name: STRING}
Vendor {partnerId: INTEGER, name: STRING, category: STRING, addcostStr: STRING, address: STRING, coverUrl: STRING, detailCmt: STRING, enterpriseCode: STRING, eventOptionPrice: INTEGER, eventPrice: INTEGER, holiday: STRING, iweddingNo: STRING, likeCnt: INTEGER, o

## 선호도 기반 추천 테스트

In [74]:
cypher_query = """
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:IN_REGION]->(r:Region)

WITH h,
     collect(DISTINCT sf.name) AS styles,
     collect(DISTINCT t.name) AS tags,
     collect(DISTINCT r.name) AS regions

WITH h, styles, tags, regions,
     CASE WHEN any(x IN styles WHERE x CONTAINS $style) THEN 2 ELSE 0 END +
     CASE WHEN any(x IN tags WHERE x CONTAINS $mood) OR h.memoContent CONTAINS $mood THEN 1 ELSE 0 END +
     CASE WHEN any(x IN regions WHERE x CONTAINS $region) THEN 2 ELSE 0 END +
     CASE WHEN any(x IN tags WHERE x CONTAINS $food) THEN 1 ELSE 0 END
     AS score

WHERE score > 0

RETURN h.partnerId AS hallId,
       h.name AS name,
       h.rating AS rating,
       h.reviewCnt AS reviewCnt,
       score
ORDER BY score DESC, rating DESC, reviewCnt DESC
LIMIT 10
"""

with driver.session() as session:
    result = session.run(
        cypher_query,
        style=str(pref.get("style", "")),
        mood=str(pref.get("mood", "")),
        food=str(pref.get("food", "")),
        region=str(pref.get("venue", pref.get("region", "")))
    )
    halls = [record.data() for record in result]

print("=== 선호 기반 추천 결과 ===")
print(halls)

=== 선호 기반 추천 결과 ===
[{'hallId': 1182, 'name': 'KDW웨딩', 'rating': 90.0, 'reviewCnt': 606, 'score': 6}, {'hallId': 12493, 'name': 'HW컨벤션센터', 'rating': 90.0, 'reviewCnt': 17, 'score': 6}, {'hallId': 4212, 'name': '서울웨딩타워', 'rating': 88.0, 'reviewCnt': 621, 'score': 6}, {'hallId': 11958, 'name': '메리스에이프럴', 'rating': 88.0, 'reviewCnt': 244, 'score': 6}, {'hallId': 1283, 'name': '엘컨벤션', 'rating': 87.0, 'reviewCnt': 59, 'score': 6}, {'hallId': 2482, 'name': '포레스트원 웨딩', 'rating': 86.0, 'reviewCnt': 115, 'score': 6}, {'hallId': 11380, 'name': '그래머시 코엑스_강남', 'rating': 86.0, 'reviewCnt': 15, 'score': 6}, {'hallId': 12113, 'name': '웨스턴베니비스 영등포점_영등포', 'rating': 85.0, 'reviewCnt': 51, 'score': 6}, {'hallId': 9652, 'name': 'BNT CONVENTION 웨딩', 'rating': 85.0, 'reviewCnt': 42, 'score': 6}, {'hallId': 3573, 'name': 'JW Marriott 동대문 스퀘어', 'rating': 85.0, 'reviewCnt': 17, 'score': 6}]


### 예시

In [75]:
examples = [
    # 2. 지역 기반 추천
    """USER INPUT: '강남 지역 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 3. 지역 + 평점
    """USER INPUT: '서울 강남구에서 평점 좋은 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)-[:IN_REGION]->(:Region {name:'서울'})
OPTIONAL MATCH (h)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 4. 스타일 기반 추천
    """USER INPUT: '호텔 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE t.name CONTAINS '호텔' OR sf.name CONTAINS '호텔' OR h.typeName CONTAINS '호텔'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 5. 분위기 기반 추천
    """USER INPUT: '밝은 분위기의 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE t.name CONTAINS '밝은'
   OR sf.name CONTAINS '밝은'
   OR h.memoContent CONTAINS '밝은'
   OR t.name CONTAINS '화이트'
   OR sf.name CONTAINS '화이트'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 6. 야외/가든 스타일 추천
    """USER INPUT: '야외 분위기 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE t.name CONTAINS '야외'
   OR t.name CONTAINS '가든'
   OR sf.name CONTAINS '야외'
   OR sf.name CONTAINS '가든'
   OR h.memoContent CONTAINS '야외'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 7. 혜택 기반 추천
    """USER INPUT: '혜택이 많은 웨딩홀 알려줘'
QUERY:
MATCH (h:Hall)-[:HAS_BENEFIT]->(b:Benefit)
RETURN h.partnerId AS hallId, h.name AS name, count(b) AS benefitCount, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY benefitCount DESC, rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 8. 유사 웨딩홀 추천
    """USER INPUT: '노보텔 앰버서더와 비슷한 분위기의 웨딩홀 추천해줘'
QUERY:
MATCH (base:Hall {name:'노보텔 앰버서더'})
OPTIONAL MATCH (base)-[:HAS_TAG]->(bt:Tag)
OPTIONAL MATCH (base)-[:HAS_STYLE_FILTER]->(bs:StyleFilter)
WITH base, collect(DISTINCT bt.name) AS baseTags, collect(DISTINCT bs.name) AS baseStyles
MATCH (rec:Hall)
WHERE rec.partnerId <> base.partnerId
OPTIONAL MATCH (rec)-[:HAS_TAG]->(rt:Tag)
OPTIONAL MATCH (rec)-[:HAS_STYLE_FILTER]->(rs:StyleFilter)
WITH rec, baseTags, baseStyles,
     count(DISTINCT CASE WHEN rt.name IN baseTags THEN rt.name END) AS tagOverlap,
     count(DISTINCT CASE WHEN rs.name IN baseStyles THEN rs.name END) AS styleOverlap
RETURN rec.partnerId AS hallId, rec.name AS recommendation,
       rec.rating AS rating, rec.reviewCnt AS reviewCnt,
       tagOverlap, styleOverlap
ORDER BY (tagOverlap * 3 + styleOverlap * 2) DESC, rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 9. 가격 비교
    """USER INPUT: '노보텔 앰버서더와 삼정호텔 비교해줘'
QUERY:
MATCH (a:Hall {name:'노보텔 앰버서더'})
MATCH (b:Hall {name:'삼정호텔'})
RETURN
a.name AS hallAName,
a.rating AS hallARating,
a.reviewCnt AS hallAReviewCnt,
a.minMealPrice AS hallAMinMealPrice,
a.maxMealPrice AS hallAMaxMealPrice,
a.minRentalPrice AS hallAMinRentalPrice,
a.maxRentalPrice AS hallAMaxRentalPrice,
b.name AS hallBName,
b.rating AS hallBRating,
b.reviewCnt AS hallBReviewCnt,
b.minMealPrice AS hallBMinMealPrice,
b.maxMealPrice AS hallBMaxMealPrice,
b.minRentalPrice AS hallBMinRentalPrice,
b.maxRentalPrice AS hallBMaxRentalPrice
""",

    # 10. 리뷰 많은 홀
    """USER INPUT: '리뷰가 많은 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.reviewCnt IS NOT NULL
RETURN h.partnerId AS hallId, h.name AS name, h.reviewCnt AS reviewCnt, h.rating AS rating
ORDER BY reviewCnt DESC, rating DESC
LIMIT 10
""",

    # 11. 평점 높은 홀
    """USER INPUT: '평점 높은 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.rating IS NOT NULL
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 12. 식대 기준 추천
    """USER INPUT: '식대가 저렴한 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.minMealPrice IS NOT NULL
RETURN h.partnerId AS hallId, h.name AS name, h.minMealPrice AS minMealPrice, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY minMealPrice ASC, rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 13. 강남 + 호텔 + 평점
    """USER INPUT: '강남에 있는 호텔 웨딩홀 중 평점 좋은 곳 추천해줘'
QUERY:
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE d.name CONTAINS '강남'
  AND (t.name CONTAINS '호텔' OR sf.name CONTAINS '호텔' OR h.typeName CONTAINS '호텔')
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
""",

    # 14. 채플 스타일 추천
    """USER INPUT: '채플 스타일 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE t.name CONTAINS '채플' OR sf.name CONTAINS '채플' OR h.memoContent CONTAINS '채플'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
""",

"""USER INPUT: '주차 가능한 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.memoContent CONTAINS '주차'
RETURN h.name, h.rating, h.reviewCnt
ORDER BY rating DESC
LIMIT 10
""",

"""USER INPUT: '발렛파킹 되는 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.memoContent CONTAINS '발렛'
RETURN h.name, h.rating, h.reviewCnt
ORDER BY rating DESC
LIMIT 10
""",

"""USER INPUT: '역세권 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.memoContent CONTAINS '역'
RETURN h.name, h.rating
ORDER BY rating DESC
LIMIT 10
""",

"""USER INPUT: '주차장 있는 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.memoContent CONTAINS '주차'
RETURN h.partnerId AS hallId,
       h.name AS name,
       h.rating AS rating,
       h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
"""
]

In [76]:
llm = OpenAILLM(
    model_name="gpt-4o",
    model_params={"temperature": 0}
)

In [77]:
def recommend_by_preference(driver, pref):
    cypher_query = """
    MATCH (h:Hall)
    OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
    OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
    OPTIONAL MATCH (h)-[:IN_REGION]->(r:Region)

    WITH h,
         collect(DISTINCT sf.name) AS styles,
         collect(DISTINCT t.name) AS tags,
         collect(DISTINCT r.name) AS regions

    WITH h, styles, tags, regions,
         CASE WHEN any(x IN styles WHERE x CONTAINS $style) THEN 2 ELSE 0 END +
         CASE WHEN any(x IN tags WHERE x CONTAINS $mood) OR h.memoContent CONTAINS $mood THEN 1 ELSE 0 END +
         CASE WHEN any(x IN regions WHERE x CONTAINS $region) THEN 2 ELSE 0 END +
         CASE WHEN any(x IN tags WHERE x CONTAINS $food) THEN 1 ELSE 0 END
         AS score

    WHERE score > 0

    RETURN h.partnerId AS hallId,
           h.name AS name,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt,
           score
    ORDER BY score DESC, rating DESC, reviewCnt DESC
    LIMIT 10
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            style=str(pref.get("style", "")),
            mood=str(pref.get("mood", "")),
            food=str(pref.get("food", "")),
            region=str(pref.get("venue", pref.get("region", ""))),
            sub_region=str(pref.get("sub_region", pref.get("sub_region", "")))
        )
        return [record.data() for record in result]

In [78]:
def recommend_by_facility(driver, keyword, region=None):

    cypher_query = """
    MATCH (h:Hall)-[:IN_REGION]->(r:Region)

    WHERE h.memoContent CONTAINS $keyword
      AND ($region IS NULL OR r.name CONTAINS $region)

    RETURN h.partnerId AS hallId,
           h.name AS name,
           r.name AS region,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt

    ORDER BY rating DESC, reviewCnt DESC
    LIMIT 10
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            keyword=keyword,
            region=region
        )

        return [record.data() for record in result]

In [79]:
def extract_region(message: str):
    msg = message.strip()

    region_patterns = [
        r"(서울|경기|인천|부산|대구|대전|광주|울산|세종|제주)\s*(지역|쪽)?",
    ]

    for pattern in region_patterns:
        m = re.search(pattern, msg)
        if m:
            return m.group(1)

    return None

In [80]:
def extract_facility_keyword(message: str) -> str:
    if "주차" in message or "주차장" in message:
        return "주차"
    if "발렛" in message or "발렛파킹" in message:
        return "발렛"
    if "역세권" in message:
        return "역"
    if "보증인원" in message:
        return "보증인원"
    if "코스" in message:
        return "코스"
    if "뷔페" in message:
        return "뷔페"
    return ""

In [81]:
def kakao_search_keyword(query: str):
    import os
    import requests

    api_key = os.environ.get("KAKAO_REST_API_KEY")
    if not api_key:
        raise ValueError("KAKAO_REST_API_KEY가 설정되지 않았습니다.")

    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {"Authorization": f"KakaoAK {api_key.strip()}"}
    params = {"query": query}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
    except requests.RequestException as e:
        print(f"[KAKAO keyword 요청 실패] query={query}, error={e}")
        return None

    if resp.status_code == 429:
        print(f"[KAKAO keyword 429] query={query}")
        return None

    if resp.status_code != 200:
        print(f"[KAKAO keyword 오류] status={resp.status_code}, query={query}")
        print(resp.text)
        return None

    data = resp.json()
    docs = data.get("documents", [])
    if not docs:
        return None

    first = docs[0]
    return {
        "x": float(first["x"]),
        "y": float(first["y"]),
        "place_name": first.get("place_name"),
        "address_name": first.get("address_name"),
    }

In [82]:
def kakao_search_address(query: str):
    import os
    import requests

    api_key = os.environ.get("KAKAO_REST_API_KEY")
    if not api_key:
        raise ValueError("KAKAO_REST_API_KEY가 설정되지 않았습니다.")

    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key.strip()}"}
    params = {"query": query}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
    except requests.RequestException as e:
        print(f"[KAKAO address 요청 실패] query={query}, error={e}")
        return None

    if resp.status_code == 429:
        print(f"[KAKAO address 429] query={query}")
        return None

    if resp.status_code != 200:
        print(f"[KAKAO address 오류] status={resp.status_code}, query={query}")
        print(resp.text)
        return None

    data = resp.json()
    docs = data.get("documents", [])
    if not docs:
        return None

    first = docs[0]
    return {
        "x": float(first["x"]),
        "y": float(first["y"]),
        "address_name": first.get("address_name"),
    }

In [83]:
geo_cache = {}

def geocode_place(query: str):
    query = (query or "").strip()
    if not query:
        return None

    if query in geo_cache:
        return geo_cache[query]

    result = kakao_search_address(query)
    if result:
        coord = (result["y"], result["x"])
        geo_cache[query] = coord
        return coord

    result = kakao_search_keyword(query)
    if result:
        coord = (result["y"], result["x"])
        geo_cache[query] = coord
        return coord

    geo_cache[query] = None
    return None

In [84]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [85]:
def search_halls_for_location(driver, region=None, districts=None, limit=100):
    query = """
    MATCH (h:Hall)
    WHERE ($region IS NULL
           OR h.region = $region
           OR h.address STARTS WITH $region)
      AND (
            size($districts) = 0
            OR any(x IN $districts WHERE h.address CONTAINS x OR h.subRegion CONTAINS x)
      )
    RETURN DISTINCT
           h.partnerId AS hallId,
           h.name AS name,
           h.address AS address,
           h.region AS region,
           h.subRegion AS district,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt
    LIMIT $limit
    """

    print("DEBUG NEW query:", query)
    print("DEBUG NEW region param:", repr(region))
    print("DEBUG NEW districts param:", districts)

    with driver.session() as session:
        result = session.run(
            query,
            region=region,
            districts=districts or [],
            limit=limit
        )
        rows = [record.data() for record in result]

    print("DEBUG NEW rows count:", len(rows))
    return rows

In [86]:
def filter_halls_by_region_and_district(halls, region=None, districts=None):
    filtered = []

    for h in halls:
        address = (h.get("address") or "").strip()
        district = (h.get("district") or "").strip()
        hall_region = (h.get("region") or "").strip()

        if region:
            if not (address.startswith(region) or hall_region == region):
                continue

        if districts:
            if not any(d in address or d in district for d in districts):
                continue

        filtered.append(h)

    return filtered

In [87]:
def infer_region_from_start_location(start_location: str):
    if not start_location:
        return None, []

    if any(x in start_location for x in ["역삼", "강남", "선릉", "삼성", "청담", "압구정", "잠실"]):
        return "서울", []

    if any(x in start_location for x in ["판교", "분당", "수원", "용인"]):
        return "경기", []

    return None, []

In [88]:
def calculate_distance_and_sort(start_location: str, halls: list[dict]):
    start_coord = geocode_place(start_location)
    if not start_coord:
        return []

    start_lat, start_lng = start_coord
    ranked = []

    for hall in halls:
        address = hall.get("address")
        if not address:
            continue

        hall_coord = geocode_place(address)
        if not hall_coord:
            continue

        hall_lat, hall_lng = hall_coord

        distance_km = haversine_distance(start_lat, start_lng, hall_lat, hall_lng)

        item = dict(hall)
        item["distance_km"] = round(distance_km, 2)
        ranked.append(item)

    ranked.sort(
        key=lambda x: (
            x["distance_km"],
            -(x.get("rating") or 0),
            -(x.get("reviewCnt") or 0)
        )
    )
    return ranked

In [89]:
def extract_start_location(message: str):
    msg = message.strip()

    patterns = [
        r"(.+?)에서\s*(가까운|근처|인근|몇\s*분|도보|차로|이내)",
        r"(.+?)기준으로",
        r"(.+?)\s*근처(?:로)?",
        r"(.+?)\s*인근",
        r"(.+?)\s*주변",
        r"(.+?)\s*가까운\s*곳",
    ]

    for pattern in patterns:
        m = re.search(pattern, msg)
        if m:
            location = m.group(1).strip()

            location = re.sub(
                r"(웨딩홀|예식장|추천|알려줘|찾아줘)$",
                "",
                location
            ).strip()

            if len(location) < 2:
                return None

            return location

    return None

In [90]:
def extract_location_with_llm(message: str) -> str | None:
    prompt = f"""
사용자 질문에서 출발 위치만 추출하세요.

규칙:
- 역, 지역, 동네, 랜드마크 등 위치만 추출
- 없으면 null 반환
- 설명하지 말고 JSON만 출력

예시

입력: 역삼역에서 가까운 웨딩홀
출력: {{"location": "역삼역"}}

입력: 강남 근처 웨딩홀
출력: {{"location": "강남"}}

입력: 서울에서 유명한 웨딩홀
출력: {{"location": "서울"}}

사용자 질문:
{message}
"""

    try:
        result = llm.invoke(prompt).content.strip()

        import json
        parsed = json.loads(result)

        location = parsed.get("location")

        if location and location.lower() != "null":
            return location.strip()

        return None

    except Exception as e:
        print("location extract error:", e)
        return None

In [91]:
def rerank_location_candidates_with_llm(message: str, halls: list[dict]) -> list[dict]:
    if not halls:
        return []

    hall_lines = []
    for idx, h in enumerate(halls, start=1):
        hall_lines.append(
            f"{idx}. 이름: {h.get('name', '')}, "
            f"주소: {h.get('address', '')}, "
            f"지역: {h.get('region', '')}, "
            f"구: {h.get('district', '')}, "
            f"평점: {h.get('rating', 0)}, "
            f"리뷰수: {h.get('reviewCnt', 0)}"
        )

    prompt = f"""
당신은 웨딩홀 추천 보조 시스템입니다.

사용자 질문:
{message}

후보 웨딩홀 목록:
{chr(10).join(hall_lines)}

작업:
1. 사용자 질문에 맞지 않는 후보는 제외하세요.
2. 위치 질문이면 같은 권역의 후보를 우선하세요.
3. 질문과 무관한 지역은 제외하세요.
4. 결과는 가장 적절한 순서대로 최대 5개만 고르세요.
5. JSON만 출력하세요.

형식:
{{
  "selected_indices": [1, 3, 5]
}}
"""
    try:
        result = llm.invoke(prompt).content.strip()
        parsed = json.loads(result)
        selected = parsed.get("selected_indices", [])

        reranked = []
        for i in selected:
            if 1 <= i <= len(halls):
                reranked.append(halls[i - 1])

        return reranked if reranked else halls[:5]

    except Exception as e:
        print("LLM rerank error:", e)
        return halls[:5]

In [92]:
def recommend_by_location(driver, conn, couple_id, message):
    start_location = extract_location_with_llm(message)
    region = extract_region(message)
    districts = []

    if not start_location or len(start_location) < 2:
        return {
            "error": "출발 위치를 이해하지 못했습니다. 예: 역삼역에서 가까운 웨딩홀 추천해줘"
        }

    if not region:
        inferred_region, inferred_districts = infer_region_from_start_location(start_location)
        region = inferred_region
        districts = inferred_districts

    halls = search_halls_for_location(driver, region=region, districts=districts, limit=100)
    halls = filter_halls_by_region_and_district(halls, region=region, districts=districts)

    if not halls:
        return {
            "error": "조건에 맞는 웨딩홀을 찾지 못했습니다."
        }

    ranked = calculate_distance_and_sort(start_location, halls)

    if ranked:
        return {
            "mode": "distance",
            "start_location": start_location,
            "region": region,
            "results": ranked[:10]
        }

    llm_ranked = rerank_location_candidates_with_llm(message, halls)

    return {
        "mode": "llm_fallback",
        "start_location": start_location,
        "region": region,
        "results": llm_ranked[:5]
    }

In [93]:
print(geocode_place("역삼역"))
print(geocode_place("서울 강남구 테헤란로"))

(37.5006744185994, 127.03646946847)
(37.504059366187, 127.047486752713)


In [94]:
def compare_two_halls(driver, hall_a: str, hall_b: str):
    cypher = """
    MATCH (a:Hall {name:$hall_a})
    MATCH (b:Hall {name:$hall_b})
    RETURN
        a.name AS hallAName,
        a.rating AS hallARating,
        a.reviewCnt AS hallAReviewCnt,
        a.minMealPrice AS hallAMinMealPrice,
        a.maxMealPrice AS hallAMaxMealPrice,
        a.minRentalPrice AS hallAMinRentalPrice,
        a.maxRentalPrice AS hallAMaxRentalPrice,
        b.name AS hallBName,
        b.rating AS hallBRating,
        b.reviewCnt AS hallBReviewCnt,
        b.minMealPrice AS hallBMinMealPrice,
        b.maxMealPrice AS hallBMaxMealPrice,
        b.minRentalPrice AS hallBMinRentalPrice,
        b.maxRentalPrice AS hallBMaxRentalPrice
    """
    with driver.session() as session:
        result = session.run(cypher, hall_a=hall_a, hall_b=hall_b)
        return [record.data() for record in result]

In [95]:
def extract_multiple_halls(message: str):
    msg = message.strip()

    # 비교 관련 표현 제거
    msg = re.sub(r"(비교해줘|비교해\s*줘|비교|알려줘|추천해줘)$", "", msg).strip()

    # 구분자 통일
    msg = re.sub(r"\s*(?:,|vs|VS|그리고|랑|하고)\s*", "|", msg)
    msg = re.sub(r"\s*(.+?)\s*와\s*(.+)", r"\1|\2", msg)
    msg = re.sub(r"\s*(.+?)\s*과\s*(.+)", r"\1|\2", msg)

    parts = [x.strip() for x in msg.split("|") if x.strip()]
    return parts

In [96]:
print(extract_multiple_halls("노보텔과 삼정호텔 비교해줘"))
# ['노보텔', '삼정호텔']

print(extract_multiple_halls("노보텔 앰버서더와 삼정호텔 비교해줘"))
# ['노보텔 앰버서더', '삼정호텔']

print(extract_multiple_halls("노보텔, 삼정호텔, 아펠가모 반포점 비교해줘"))
# ['노보텔', '삼정호텔', '아펠가모 반포점']

['노보텔', '삼정호텔']
['노보텔 앰버서더', '삼정호텔']
['노보텔', '삼정호텔', '아펠가모 반포점']


In [97]:
def resolve_hall_name(driver, keyword: str):
    cypher = """
    MATCH (h:Hall)
    WHERE h.name CONTAINS $keyword
       OR replace(h.name, " ", "") CONTAINS replace($keyword, " ", "")
    RETURN h.name AS name,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt
    ORDER BY h.reviewCnt DESC, h.rating DESC
    LIMIT 1
    """
    with driver.session() as session:
        result = session.run(cypher, keyword=keyword)
        row = result.single()
        return row["name"] if row else None

In [98]:
def resolve_multiple_hall_names(driver, hall_keywords):
    resolved = []

    for keyword in hall_keywords:
        real_name = resolve_hall_name(driver, keyword)
        if real_name:
            resolved.append(real_name)

    # 중복 제거
    resolved = list(dict.fromkeys(resolved))
    return resolved

In [99]:
def compare_multiple_halls(driver, hall_names):
    cypher = """
    MATCH (h:Hall)
    WHERE h.name IN $hall_names
    RETURN
        h.name AS name,
        h.region AS region,
        h.subRegion AS subRegion,
        h.rating AS rating,
        h.reviewCnt AS reviewCnt,
        h.minMealPrice AS minMealPrice,
        h.maxMealPrice AS maxMealPrice,
        h.minRentalPrice AS minRentalPrice,
        h.maxRentalPrice AS maxRentalPrice
    ORDER BY h.reviewCnt DESC, h.rating DESC
    """
    with driver.session() as session:
        result = session.run(cypher, hall_names=hall_names)
        return [record.data() for record in result]

In [100]:
def compare_multiple_halls(driver, hall_names):
    cypher = """
    MATCH (h:Hall)
    WHERE h.name IN $hall_names
    RETURN
        h.name AS name,
        h.region AS region,
        h.subRegion AS subRegion,
        h.rating AS rating,
        h.reviewCnt AS reviewCnt,
        h.minMealPrice AS minMealPrice,
        h.maxMealPrice AS maxMealPrice,
        h.minRentalPrice AS minRentalPrice,
        h.maxRentalPrice AS maxRentalPrice
    """

    with driver.session() as session:
        result = session.run(cypher, hall_names=hall_names)
        return [record.data() for record in result]

In [101]:
def make_multi_compare_answer(rows):
    if not rows:
        return "비교할 웨딩홀 정보를 찾지 못했습니다."

    lines = ["웨딩홀 비교 결과입니다.\n"]

    for idx, row in enumerate(rows, start=1):
        lines.append(
            f"{idx}. {row['name']}\n"
            f"- 지역: {row.get('region', '정보없음')} {row.get('subRegion', '')}\n"
            f"- 평점: {row.get('rating', '정보없음')}\n"
            f"- 리뷰수: {row.get('reviewCnt', '정보없음')}\n"
            f"- 식대: {row.get('minMealPrice', '정보없음')} ~ {row.get('maxMealPrice', '정보없음')}\n"
            f"- 대관료: {row.get('minRentalPrice', '정보없음')} ~ {row.get('maxRentalPrice', '정보없음')}\n"
        )

    return "\n".join(lines)

## LLM / Retriever / GraphRAG

In [102]:
retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples,
)

rag = GraphRAG(
    retriever=retriever,
    llm=llm
)

### Retriever / RAG 테스트

In [103]:
query_text = "서울 강남의 괜찮은 웨딩홀 추천해줘"
search_result = retriever.search(query_text=query_text)

print("=== 생성된 Cypher ===")
print(search_result.metadata.get("cypher", ""))

print("=== 조회 결과 ===")
print(search_result.items)

query_text = "더리버사이드호텔 웨딩과 비슷한 웨딩홀 중 평점이 높은 웨딩홀은 어디인가요?"
response = rag.search(query_text=query_text, return_context=True)

print("=== RAG 생성 Cypher ===")
print(response.retriever_result.metadata.get("cypher", ""))

print("=== 최종 답변 ===")
print(response.answer)

=== 생성된 Cypher ===
MATCH (h:Hall)-[:IN_REGION]->(:Region {name:'서울'})
OPTIONAL MATCH (h)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY rating DESC, reviewCnt DESC
LIMIT 10
=== 조회 결과 ===
[RetrieverResultItem(content="<Record hallId=1182 name='KDW웨딩' rating=90.0 reviewCnt=606>", metadata=None), RetrieverResultItem(content="<Record hallId=12493 name='HW컨벤션센터' rating=90.0 reviewCnt=17>", metadata=None), RetrieverResultItem(content="<Record hallId=4212 name='서울웨딩타워' rating=88.0 reviewCnt=621>", metadata=None), RetrieverResultItem(content="<Record hallId=11958 name='메리스에이프럴' rating=88.0 reviewCnt=244>", metadata=None), RetrieverResultItem(content="<Record hallId=1283 name='엘컨벤션' rating=87.0 reviewCnt=59>", metadata=None), RetrieverResultItem(content="<Record hallId=2482 name='포레스트원 웨딩' rating=86.0 reviewCnt=115>", metadata=None), RetrieverResultItem(content="<Record hallId=11380 name

### Gradio

In [104]:
def safe_str(x, max_len=6000):
    try:
        s = json.dumps(x, ensure_ascii=False, indent=2)
    except Exception:
        s = str(x)
    return s if len(s) <= max_len else s[:max_len] + "\n... (truncated)"


class Seafoam(gr.themes.Base):
    pass


seafoam = Seafoam()

In [105]:
raw_halls = search_halls_for_location(driver, region="서울", districts=[], limit=10)
print(type(raw_halls))
print(len(raw_halls))
for h in raw_halls[:5]:
    print(h)

DEBUG NEW query: 
    MATCH (h:Hall)
    WHERE ($region IS NULL
           OR h.region = $region
           OR h.address STARTS WITH $region)
      AND (
            size($districts) = 0
            OR any(x IN $districts WHERE h.address CONTAINS x OR h.subRegion CONTAINS x)
      )
    RETURN DISTINCT
           h.partnerId AS hallId,
           h.name AS name,
           h.address AS address,
           h.region AS region,
           h.subRegion AS district,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt
    LIMIT $limit
    
DEBUG NEW region param: '서울'
DEBUG NEW districts param: []
DEBUG NEW rows count: 10
<class 'list'>
10
{'hallId': 14117, 'name': '풀만 앰배서더 이스트폴 웨딩', 'address': '서울 광진구', 'region': '서울', 'district': '광진구', 'rating': 0.0, 'reviewCnt': None}
{'hallId': 1438, 'name': '그랜드 머큐어 임피리얼 팰리스 서울_강남', 'address': '서울 강남구', 'region': '서울', 'district': '강남구', 'rating': 76.0, 'reviewCnt': 22}
{'hallId': 1437, 'name': '더화이트베일_서초', 'address': '서울 서초구', 'region': 

In [106]:
result = recommend_by_location(driver, conn, COUPLE_ID, "역삼역에서 가까운 웨딩홀 추천해줘")
print(result)

DEBUG NEW query: 
    MATCH (h:Hall)
    WHERE ($region IS NULL
           OR h.region = $region
           OR h.address STARTS WITH $region)
      AND (
            size($districts) = 0
            OR any(x IN $districts WHERE h.address CONTAINS x OR h.subRegion CONTAINS x)
      )
    RETURN DISTINCT
           h.partnerId AS hallId,
           h.name AS name,
           h.address AS address,
           h.region AS region,
           h.subRegion AS district,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt
    LIMIT $limit
    
DEBUG NEW region param: '서울'
DEBUG NEW districts param: []
DEBUG NEW rows count: 100
{'mode': 'distance', 'start_location': '역삼역', 'region': '서울', 'results': [{'hallId': 760, 'name': '더리버사이드호텔 웨딩', 'address': '서울 서초구', 'region': '서울', 'district': '서초구', 'rating': 80.0, 'reviewCnt': 688, 'distance_km': 1.93}, {'hallId': 1237, 'name': '아펠가모 반포점', 'address': '서울 서초구', 'region': '서울', 'district': '서초구', 'rating': 79.0, 'reviewCnt': 245, 'distance

### Gradio App

In [ ]:
with gr.Blocks(theme=seafoam) as demo:

    def classify_intent(message: str) -> str:
        msg = message.strip()

        if msg == "웨딩홀 추천해줘":
            return "PREFERENCE_RECOMMEND"
        
        # 5. 위치 기반 질문
        if any(x in msg for x in ["가까운", "거리", "몇 분", "역에서", "이내", "도보", "차로", "근처", "인근"]):
            return "LOCATION"
        
        if any(x in msg for x in ["주차", "주차장", "발렛", "발렛파킹", "역세권", "보증인원", "코스", "뷔페"]):
            return "FACILITY_SEARCH"

        # 1. 좋아요 기반 추천
        if any(x in msg for x in ["좋아요한 웨딩홀과 비슷", "찜한 웨딩홀과 비슷", "좋아요 누른 웨딩홀과 비슷"]):
            return "LIKE_BASED_RECOMMEND"

        # 2. 좋아요 조회
        if any(x in msg for x in ["좋아요 누른", "좋아요한", "찜한 웨딩홀", "내가 찜한", "내가 좋아한"]):
            return "LIKE_QUERY"

        # 3. 취향 조회/취향 기반 추천
        if any(x in msg for x in ["내 취향", "내 스타일", "내 지역", "내 예산", "내가 선택"]):
            return "PREFERENCE_QUERY"

        # 4. 비교 질문
        if any(x in msg for x in ["비교", "차이", "어디가 더", "중 뭐가", "중 어디가", "vs", "VS"]):
            return "COMPARE"

        # 6. 일반 웨딩홀 추천/검색
        if any(x in msg for x in ["웨딩홀", "예식장", "추천", "스타일", "분위기", "혜택", "평점", "리뷰", "호텔", "채플", "야외"]):
            return "NEO4J_RAG"
        

        return "GENERAL"
        
    def default_llm(message: str) -> str:
        prompt_text = f"""
당신은 웨딩 전문 챗봇입니다. 사용자의 질문에 친절하게 답해주세요.

[답변 가이드라인]
1. 사용자가 스타일, 트렌드, 분위기를 물어보면 구체적인 예시를 들어 설명해주세요.
2. 가격만 강조하지 말고 분위기, 위치, 후기, 식사 등도 함께 설명해주세요.
3. 답변 끝에는 자연스럽게 추가 질문을 유도해주세요.

user_input: {message}
"""
        return llm.invoke(prompt_text).content

    def response_fn(message, chat_history):
        chat_history = chat_history or []
        intent = classify_intent(message)

        if intent == "PREFERENCE_QUERY":
            user_pref = get_user_preference(conn, COUPLE_ID)

            if user_pref is None:
                answer = "저장된 취향 정보가 없습니다."
                items = ""
            else:
                answer = (
                    f"현재 저장된 취향 정보입니다.\n"
                    f"- 지역: {user_pref.get('venue', user_pref.get('region', '없음'))}\n"
                    f"- 세부지역: {user_pref.get('sub_region', '없음')}\n"
                    f"- 스타일: {user_pref.get('style', '없음')}\n"
                    f"- 분위기: {user_pref.get('mood', '없음')}\n"
                    f"- 음식: {user_pref.get('food', '없음')}\n"
                    f"- 예산: {user_pref.get('budget', '없음')}"
                )
                items = safe_str(user_pref)

            cypher = "MySQL - COUPLE_PREFERENCES"

        elif intent == "PREFERENCE_RECOMMEND":
            user_pref = get_user_preference(conn, COUPLE_ID)

            if user_pref is None:
                answer = "저장된 취향 정보가 없어 기본 추천을 진행할 수 없습니다."
                items = ""
                cypher = "COUPLE_PREFERENCES 없음"
            else:
                halls = recommend_by_preference(driver, user_pref)

                if not halls:
                    answer = "현재 저장된 취향과 일치하는 웨딩홀을 찾지 못했습니다."
                    items = ""
                else:
                    lines = []
                    for h in halls[:5]:
                        lines.append(
                            f"- {h['name']} (평점 {h['rating']}, 리뷰 {h['reviewCnt']}, 추천점수 {h['score']})"
                        )

                    answer = "저장된 취향을 바탕으로 추천한 웨딩홀입니다.\n\n" + "\n".join(lines)
                    items = safe_str(halls)

                cypher = "Preference-based Cypher"

        elif intent == "LIKE_QUERY":
            likes = get_user_likes(conn, COUPLE_ID)

            if not likes:
                answer = "좋아요한 웨딩홀이 없습니다."
                items = ""
            else:
                answer = f"좋아요한 웨딩홀 기록이 {len(likes)}건 있습니다."
                items = safe_str(likes)

            cypher = "MySQL - COUPLE_VENUE_LIKES"

        elif intent == "LIKE_BASED_RECOMMEND":
            user_pref = get_user_preference(conn, COUPLE_ID)

            preferred_region = None
            preferred_sub_region = None

            if user_pref:
                preferred_region = user_pref.get("region") or user_pref.get("venue")
                preferred_sub_region = user_pref.get("sub_region")

            rec = recommend_by_likes_with_region_boost(
                conn,
                COUPLE_ID,
                preferred_region=preferred_region,
                preferred_sub_region=preferred_sub_region
            )

            if not rec:
                answer = "좋아요와 지역 정보를 기반으로 한 추천 결과가 없습니다."
                items = ""
            else:
                lines = []
                for r in rec:
                    lines.append(
                        f"- {r['name']} ({r['region']} {r['sub_region']}, 평점 {r['rating']})"
                    )

                answer = (
                    "좋아요 기록과 선호 지역을 함께 반영한 추천 결과입니다.\n\n"
                    + "\n".join(lines)
                )
                items = safe_str(rec)

            cypher = "MySQL Collaborative Filtering + Region"

        elif intent == "FACILITY_SEARCH":
            keyword = extract_facility_keyword(message)
            region = extract_region(message)

            # 질문에 지역 없으면 취향 지역 사용
            if region is None:
                user_pref = get_user_preference(conn, COUPLE_ID)

                if user_pref:
                    region = user_pref.get("region") or user_pref.get("venue")

            halls = recommend_by_facility(driver, keyword, region)

            if not halls:
                answer = f"'{keyword}' 관련 웨딩홀을 찾지 못했습니다."
                items = ""
            else:
                lines = []
                for h in halls[:5]:
                    lines.append(
                        f"- {h['name']} (평점 {h['rating']}, 리뷰 {h['reviewCnt']})"
                    )

                if region:
                    answer = f"{region} 지역에서 '{keyword}' 정보를 기준으로 추천한 웨딩홀입니다.\n\n" + "\n".join(lines)
                else:
                    answer = f"'{keyword}' 정보를 기준으로 추천한 웨딩홀입니다.\n\n" + "\n".join(lines)

                items = safe_str(halls)

            cypher = f"Facility Search: memoContent CONTAINS '{keyword}', region={region}"

        elif intent == "COMPARE":
            hall_keywords = extract_multiple_halls(message)
            print("DEBUG hall_keywords:", hall_keywords)

            resolved_names = resolve_multiple_hall_names(driver, hall_keywords)
            print("DEBUG resolved_names:", resolved_names)

            if len(resolved_names) < 2:
                answer = "비교할 웨딩홀 이름을 두 개 이상 찾지 못했습니다."
                cypher = "COMPARE resolve failed"
                items = safe_str({
                    "input_keywords": hall_keywords,
                    "resolved_names": resolved_names
                })
            else:
                rows = compare_multiple_halls(driver, resolved_names)

                if len(rows) < 2:
                    answer = "비교할 웨딩홀 정보를 충분히 찾지 못했습니다."
                    items = safe_str(rows)
                else:
                    answer = make_multi_compare_answer(rows)
                    items = safe_str(rows)

                cypher = "MULTI HALL COMPARE QUERY"

        elif intent == "LOCATION":
            result = recommend_by_location(driver, conn, COUPLE_ID, message)

            if result.get("error"):
                answer = result["error"]
                items = ""
                cypher = "Location search failed"
            else:
                halls = result["results"]
                start_location = result["start_location"]
                region = result["region"]
                mode = result.get("mode", "distance")

                if not halls:
                    answer = f"{start_location} 기준으로 가까운 웨딩홀을 찾지 못했습니다."
                    items = ""
                else:
                    lines = []
                    for h in halls[:5]:
                        lines.append(
                            f"- {h['name']} ({h.get('address', '주소없음')})"
                        )

                    answer = f"{start_location} 기준으로 접근성이 좋아 보이는 웨딩홀입니다.\n\n" + "\n".join(lines)

                    items = safe_str(halls)

                cypher = f"Location search start={start_location}, region={region}, mode={mode}"
                
        elif intent == "NEO4J_RAG":
            rag_result = rag.search(query_text=message, return_context=True)
            answer = rag_result.answer
            cypher = rag_result.retriever_result.metadata.get("cypher", "")
            items = safe_str(rag_result.retriever_result.items)

        else:
            answer = default_llm(message)
            cypher = "일반 대화"
            items = "일반 대화"

        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": answer})

        return chat_history, cypher, items

    with gr.Row():
        gr.HTML("""
        <div style="text-align:center; max-width:1000px; margin:10px auto;">
            <h1>Graph RAG 웨딩홀 챗봇</h1>
            <p>Neo4j 그래프 DB 기반으로 웨딩홀을 추천하고, 생성된 Cypher와 조회 결과도 함께 보여줍니다.</p>
        </div>
        """)

    with gr.Row():
        with gr.Column(scale=1):
            generated_query = gr.Textbox(label="생성된 Cypher 쿼리", lines=18)
            query_result = gr.Textbox(label="쿼리 조회 결과", lines=18)

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type="messages", )
            msg = gr.Textbox(
                placeholder="예: 서울 강남구에서 평점 좋은 웨딩홀 추천해줘",
                label="입력"
            )

            gr.Examples(
                examples=[
                    "저는 노보텔 앰버서더와 같은 예식장을 좋아합니다. 이 예식장과 비슷한 웨딩홀을 추천해줄 수 있나요?",
                    "웨딩홀 아펠가모 광화문점과 비슷한 혹은 채플 스타일 웨딩홀을 추천해주세요.",
                    "서울 강남구에서 평점과 리뷰가 좋은 웨딩홀 5개 추천해줘",
                    "내 취향이 뭐로 저장돼 있어?",
                    "내가 좋아요 누른 웨딩홀 보여줘",
                    "좋아요 누른 웨딩홀과 비슷한 곳 추천해줘",
                    "역삼역에서 가까운 웨딩홀 추천해줘"
                ],
                inputs=[msg]
            )

            with gr.Row():
                btn = gr.Button("Submit", variant="primary")
                clear = gr.Button("Clear")

    submit_event = dict(
        fn=response_fn,
        inputs=[msg, chatbot],
        outputs=[chatbot, generated_query, query_result]
    )

    btn.click(**submit_event).then(lambda: "", None, msg)
    msg.submit(**submit_event).then(lambda: "", None, msg)

    def clear_all():
        return [], "", ""

    clear.click(
        fn=clear_all,
        inputs=None,
        outputs=[chatbot, generated_query, query_result],
        queue=False
    ).then(lambda: "", None, msg)

demo.launch(debug=True, share=True)

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_26648\1368167436.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=seafoam) as demo:
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_26648\1368167436.py:265: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", )


* Running on local URL:  http://127.0.0.1:7864

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/03/16 09:47:07 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout
